# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a walkthrough for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

This dataset details the analysis of protein abundance, modifications, and sequences in human (Homo sapiens) samples. It includes fields such as accession, description, coverage percentage, peptide counts, MW, calculated parameters such as pI, and normalized abundances across different samples. Data was collected and processed using mass spectrometry.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. The Croissant schema will provide details about available record sets and fields.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
ds_metadata = dataset.metadata
print("Dataset Name:", ds_metadata.name)
print("Description:", ds_metadata.description)
print("Publication Date:", getattr(ds_metadata, 'datePublished', 'N/A'))
print("License:", getattr(ds_metadata, 'license', 'N/A'))

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

A record set is the main table or structured collection in the dataset, each with fields (columns) that have unique `@id` values.

In [ ]:
record_sets = list(dataset.metadata.record_sets)
print(f"There are {len(record_sets)} record sets available.")
for rs in record_sets:
    print(f"RecordSet name: {getattr(rs, 'name', 'N/A')} | @id: {rs.id}")
    print("  Fields:")
    for fld in rs.fields:
        print(f"    - Field: {getattr(fld, 'name', 'N/A')} | @id: {fld.id} | dataType: {getattr(fld, 'data_type', 'N/A')}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

We'll choose the main data record set for proteins (using its `@id`).

_Note: Replace `record_set_proteins_id` below with the appropriate `@id` from the overview above._

In [ ]:
# Find the main protein table (assume the first record set if one exists)
if not record_sets:
    raise RuntimeError("No record sets found in the metadata.")
# Usually the first is the main table
main_rs = record_sets[0]
record_set_proteins_id = main_rs.id  # e.g. 'cr:RecordSet_1'
print(f"Loading records from RecordSet: {record_set_proteins_id}")

df_records = list(dataset.records(record_set=record_set_proteins_id))
df = pd.DataFrame(df_records)
print(f"Loaded {len(df)} records. Columns available:")
pprint.pprint(list(df.columns))

# Display the first few records
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering, normalizing, grouping, etc.

_Suppose the protein table contains a numeric field for protein molecular weight ("MW") and grouping by a sample or category is relevant._

### Find a numeric field

We'll look for the first integer or float field (e.g., `MW` for molecular weight).

We will also select a grouping field (if one exists, e.g., `Sample` or `Category`).

In [ ]:
# Identify a numeric field and a grouping field
protein_fields = main_rs.fields

numeric_type_names = {'schema:Float', 'schema:Number', 'schema:Integer'}
numeric_field = None
group_field = None
for f in protein_fields:
    if getattr(f, 'data_type', None) in numeric_type_names and numeric_field is None:
        numeric_field = f.id  # Use @id
    # Find a likely grouping field ('sample', 'category', 'type', etc. in name)
    if group_field is None and any(x in f.name.lower() for x in ["group", "category", "sample", "type"]):
        group_field = f.id

print(f"Numeric field selected: {numeric_field}")
print(f"Grouping field selected: {group_field}")

# Defensive: ensure the field names actually map to DataFrame columns
field_id_to_col = {fld.id: fld.name for fld in protein_fields}
num_col = field_id_to_col.get(numeric_field, numeric_field)
grp_col = field_id_to_col.get(group_field, group_field) if group_field else None

# Prepare data (drop missing)
if num_col in df.columns:
    df_num = pd.to_numeric(df[num_col], errors='coerce')
    filtered_df = df[df_num > df_num.mean()]  # Example: proteins heavier than mean MW
    filtered_df = filtered_df.copy()
    print(f"Filtered records with {num_col} > mean ({df_num.mean():.2f}): {len(filtered_df)} records")
    filtered_df[f"{num_col}_normalized"] = (df_num - df_num.mean())/df_num.std()
    print(filtered_df[[num_col, f"{num_col}_normalized"]].head())
else:
    print(f"Numeric field '{num_col}' not found in table columns: {df.columns}")

# Group by group_col if it exists
if grp_col and grp_col in df.columns:
    grouped = filtered_df.groupby(grp_col)[num_col].mean().reset_index()
    print(f"Mean {num_col} by {grp_col}:")
    print(grouped.head())

## 5. Visualization

Visualize numeric distributions (e.g. histogram of MW or protein abundance), or relationships (e.g. boxplot by sample or category).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram for numeric field
if num_col in df.columns:
    plt.figure(figsize=(7,4))
    df_num = pd.to_numeric(df[num_col], errors='coerce')
    sns.histplot(df_num.dropna(), bins=30, kde=True)
    plt.xlabel(field_id_to_col.get(numeric_field, numeric_field))
    plt.title(f"Distribution of {num_col}")
    plt.show()

# Boxplot of MW/group, if available
if grp_col and grp_col in df.columns and num_col in df.columns:
    plt.figure(figsize=(10,4))
    sns.boxplot(x=df[grp_col], y=pd.to_numeric(df[num_col], errors='coerce'))
    plt.xlabel(grp_col)
    plt.ylabel(num_col)
    plt.title(f"{num_col} by {grp_col}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion

This notebook demonstrates step-by-step exploration of the FAIR^2 dataset using `mlcroissant`, including data loading, record set and field enumeration (always by `@id`), extraction to pandas DataFrames, basic numeric analysis, and visualization.

You can adapt these steps for deeper or domain-specific analyses.

**Key observations:**
- The dataset provides mass spectrometry-derived evidence of protein abundance and modification in human extracellular vesicles.
- Structured access via Croissant allows programmatic loading, filtering, grouping, and visualization—all referencing fields and record sets with their stable `@id`s for reproducibility.

For more advanced analytics, see the additional documentation at [`https://mlcommons.org/croissant`](https://mlcommons.org/croissant) and the [dataset landing page](https://sen.science/doi/10.71728/senscience.vq4a-28xa).